# CovIntervene frozen P0 analysis

Run this CPU-only notebook **only after both full backbone notebooks report 12/12 units**. It performs no model inference. It verifies every array hash, requires the complete frozen inventory, and computes the predeclared continuation decision once. If a decision already exists, it is validated and displayed without recomputation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
print('Repository commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import os
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements/colab-base.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import covfaith
print('covfaith imported from:', covfaith.__file__)

In [ ]:
import hashlib
import json
import yaml

from covfaith.config import load_yaml, verify_config_lock
from covfaith.p0 import scientific_code_hash

EXPECTED_CONFIG_HASH = '56cd2ff3a3e56155074a47abb02a859be5e107a6fb59679f9f3d3981110db0dd'
EXPECTED_SCIENTIFIC_CODE_SHA256 = 'ee75398fe2b7c93ac2e2fa83b0abeeb76f346a8953bfc7c91e1a1cfbd6b5bcde'
config_path = REPO / 'configs/p0/covintervene_p0.yaml'
lock_path = REPO / 'configs/p0/covintervene_p0.lock.json'
config_hash = verify_config_lock(config_path, lock_path)
code_hash = scientific_code_hash(REPO)
assert config_hash == EXPECTED_CONFIG_HASH
assert code_hash == EXPECTED_SCIENTIFIC_CODE_SHA256
test_env = os.environ.copy()
test_env['PYTHONPATH'] = src_path
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(REPO / 'tests')], cwd=REPO, env=test_env, check=True)
print('Frozen config and scientific code verified:', config_hash[:12], code_hash[:12])

In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p0_full_v1')
config = load_yaml(config_path)
verified_units = []
for model in config['models']:
    backbone = model['id']
    completion_path = OUTPUT_ROOT / f'{backbone}_full_screening_completion.json'
    completion = json.loads(completion_path.read_text(encoding='utf-8'))
    assert completion['config_hash'] == EXPECTED_CONFIG_HASH
    assert completion['scientific_code_sha256'] == EXPECTED_SCIENTIFIC_CODE_SHA256
    assert completion['mode'] == 'full_screening'
    assert completion['series_per_mechanism_per_seed'] == 64
    assert completion['completed_unit_count'] == 12
    assert completion['scientific_gate_computed'] is False
    for mechanism in config['data']['mechanisms']:
        unit_dir = OUTPUT_ROOT / 'units' / backbone / mechanism
        for seed in config['data']['generator_seeds']:
            stem = f'seed_{int(seed):05d}'
            array_path = unit_dir / f'{stem}.npz'
            manifest_path = unit_dir / f'{stem}.json'
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            assert manifest['completed'] is True
            assert manifest['mode'] == 'full_screening'
            assert manifest['series_count'] == 64
            assert manifest['config_hash'] == EXPECTED_CONFIG_HASH
            assert manifest['scientific_code_sha256'] == EXPECTED_SCIENTIFIC_CODE_SHA256
            actual_sha256 = hashlib.sha256(array_path.read_bytes()).hexdigest()
            assert actual_sha256 == manifest['array_sha256']
            verified_units.append(f'{backbone}/{mechanism}/{stem}')
assert len(verified_units) == 24
print('Verified complete immutable inventory:', len(verified_units), 'units')

In [ ]:
from covfaith.p0 import analyze_complete_p0

decision_path = OUTPUT_ROOT / 'p0_phenomenon_decision.yaml'
if decision_path.exists():
    report = yaml.safe_load(decision_path.read_text(encoding='utf-8'))
    assert report['config_hash'] == EXPECTED_CONFIG_HASH
    assert report['scientific_code_sha256'] == EXPECTED_SCIENTIFIC_CODE_SHA256
    assert report['result_status'] == 'verified_screening_decision'
    print('Existing frozen decision found; it was not recomputed.')
else:
    report = analyze_complete_p0(REPO, OUTPUT_ROOT)
    print('Frozen P0 decision computed for the first time.')
print(yaml.safe_dump(report, sort_keys=False, allow_unicode=True))
print('Decision artifact:', decision_path)